## 1. Importing Libraries

In [ ]:
# --- Data wrangling ---
import pandas as pd
import numpy as np        

# --- Visualisation ---
import matplotlib.pyplot as plt   
import seaborn as sns 

# --- Utilities ---
import warnings
import os

# --- ML: model building & evaluation ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    average_precision_score,   # PR-AUC
    precision_recall_curve,
    roc_auc_score,
    f1_score,
)

# --- Imbalanced-class handling ---
from imblearn.pipeline import Pipeline   
from imblearn.over_sampling import SMOTE

# --- Gradient-boosted tree model ---
import xgboost as xgb

# --- Explainability ---
import shap

# --- Concept drift detection ---
from river import drift

# --- Model persistence ---
import joblib

print("All libraries imported successfully.")

## 2. Global Configuration

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── File path ────────────────────────────────────────────────────────────────

DATA_PATH = "data.csv"  

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)         # Show all columns in tables
pd.set_option("display.float_format", "{:.4f}".format)  # 4 decimal places
pd.set_option("display.max_rows", 60)

warnings.filterwarnings("ignore")  

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline   

FRAUD_PALETTE = {0: "#2196F3", 1: "#F44336"}   # Blue = legit, Red = fraud

print("Configuration set.")


## 3. Define Column Data Types BEFORE Loading


In [ ]:
DTYPES = {
    "step":            "int32",     # Hour counter 1–744; int32 is plenty
    "type":            "category",  # 5 unique values → category is very efficient
    "amount":          "float32",   # Transaction value; float32 precision is fine
    "nameOrig":        "object",    # Account IDs — kept as strings for now
    "oldbalanceOrg":   "float32",
    "newbalanceOrig":  "float32",
    "nameDest":        "object",
    "oldbalanceDest":  "float32",
    "newbalanceDest":  "float32",
    "isFraud":         "int8",      # 0 or 1 only; int8 uses 1 byte vs int64's 8
    "isFlaggedFraud":  "int8",
}

print("dtypes dictionary ready. These will be passed to pd.read_csv().")

## 4. Load the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, dtype=DTYPES)

# Quick sanity report
n_rows, n_cols = df.shape
mem_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"\n Dataset loaded successfully!")
print(f"   Rows    : {n_rows:>12,}")
print(f"   Columns : {n_cols:>12}")
print(f"   RAM used: {mem_mb:>11.1f} MB")


In [ ]:
print(f"Shape: {df.shape}\n")
df.head(10)

## 5. Data Exploration

In [ ]:
print("Column types:\n")
print(df.dtypes)

In [ ]:
mem_by_col = (df.memory_usage(deep=True) / 1024**2).sort_values(ascending=False)
mem_by_col.index.name = "column"
mem_df = mem_by_col.reset_index()
mem_df.columns = ["column", "size_MB"]
mem_df["size_MB"] = mem_df["size_MB"].round(3)
print("Memory usage per column (MB):\n")
print(mem_df.to_string(index=False))

In [ ]:
df.info(memory_usage="deep")

In [ ]:
missing = df.isnull().sum()
missing_cols = missing[missing > 0]

if missing_cols.empty:
    print("No missing values in any column.")
else:
    print("Missing values found:\n")
    print(missing_cols)

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes:,}")
if n_dupes == 0:
    print("No duplicates found.")

In [ ]:
df.describe()

In [ ]:
type_counts    = df["type"].value_counts()
type_pct       = df["type"].value_counts(normalize=True).mul(100).round(2)
type_summary   = pd.DataFrame({"Count": type_counts, "Percentage (%)": type_pct})

print("Transaction Type Distribution:\n")
print(type_summary.to_string())

# Visual
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="type", order=type_counts.index,
              palette="Blues_r", ax=ax)
ax.set_title("Transaction Volume by Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Transaction Type")
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

In [ ]:
fraud_counts = df["isFraud"].value_counts()
fraud_pct    = df["isFraud"].value_counts(normalize=True).mul(100)

print("Class Distribution (isFraud):\n")
summary = pd.DataFrame({
    "Count":       fraud_counts,
    "Percentage":  fraud_pct.map("{:.4f}%".format),
    "Label":       ["Legitimate", "Fraud"],
})
print(summary.to_string())

naive_accuracy = fraud_counts[0] / fraud_counts.sum() * 100
print(f"\n A 'never-fraud' model scores {naive_accuracy:.3f}% accuracy — useless!")

# Visual
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=["Legitimate (0)", "Fraud (1)"],
            y=fraud_counts.values,
            palette=list(FRAUD_PALETTE.values()), ax=ax)
ax.set_title("Class Imbalance: isFraud", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.2f}M"))

# Add percentage labels on bars
for bar, pct in zip(ax.patches, fraud_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 0.5,
            f"{pct:.3f}%", ha="center", va="center",
            color="white", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
total_fraud   = int(df["isFraud"].sum())
total_flagged = int(df["isFlaggedFraud"].sum())
both_caught   = int(((df["isFraud"] == 1) & (df["isFlaggedFraud"] == 1)).sum())
missed        = total_fraud - both_caught
rule_recall   = both_caught / total_fraud * 100

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  isFraud vs isFlaggedFraud — Rule Audit")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Real frauds (isFraud = 1)       : {total_fraud:>7,}")
print(f"  Rule triggers (isFlaggedFraud=1): {total_flagged:>7,}")
print(f"  Frauds the rule caught          : {both_caught:>7,}")
print(f"  Frauds the rule MISSED          : {missed:>7,}")
print(f"  Rule recall                     : {rule_recall:>6.2f}%")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"\n  ← This is why we need ML.")

In [ ]:
fraud_by_type = (
    df.groupby("type", observed=True)["isFraud"]
    .agg(fraud_count="sum", total_count="count")
    .assign(fraud_rate_pct=lambda x: (x["fraud_count"] / x["total_count"] * 100).round(4))
    .sort_values("fraud_count", ascending=False)
)

print("Fraud by Transaction Type:\n")
print(fraud_by_type.to_string())

# Visual — stacked bar: fraud vs legitimate per type
type_fraud_pivot = (
    df.groupby(["type", "isFraud"], observed=True)
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "Legitimate", 1: "Fraud"})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute counts (log scale so fraud bars are visible)
type_fraud_pivot.plot(kind="bar", stacked=True,
                      color=["#2196F3", "#F44336"],
                      ax=axes[0], logy=True)
axes[0].set_title("Transaction Count by Type (log scale)", fontweight="bold")
axes[0].set_xlabel("Transaction Type")
axes[0].set_ylabel("Count (log scale)")
axes[0].tick_params(axis="x", rotation=30)

# Right: fraud rate % per type (clearer for the key insight)
fraud_by_type["fraud_rate_pct"].plot(kind="bar",
                                      color="#F44336",
                                      ax=axes[1])
axes[1].set_title("Fraud Rate % by Transaction Type", fontweight="bold")
axes[1].set_xlabel("Transaction Type")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("All Fraud is in TRANSFER and CASH_OUT", fontsize=14,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("Amount statistics by fraud label:\n")
amount_stats = df.groupby("isFraud")["amount"].describe().round(2)
amount_stats.index = amount_stats.index.map({0: "Legitimate (0)", 1: "Fraud (1)"})
print(amount_stats.to_string())

# Visual — log-scaled histogram overlay
fig, ax = plt.subplots(figsize=(9, 4))
for label, color in FRAUD_PALETTE.items():
    subset = df[df["isFraud"] == label]["amount"]
    ax.hist(subset, bins=80, alpha=0.55, color=color,
            label=f"{'Fraud' if label else 'Legitimate'} (n={len(subset):,})",
            density=True)
ax.set_xscale("log")
ax.set_title("Transaction Amount Distribution (log scale)", fontweight="bold")
ax.set_xlabel("Amount (log scale)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(f"Step range : {df['step'].min()} → {df['step'].max()}")
print(f"Unique steps: {df['step'].nunique()} (expected 744 for a 30-day sim)")
print(f"Equivalent to: {df['step'].max() / 24:.1f} simulated days")

# Fraud counts across time
hourly = df.groupby("step")["isFraud"].agg(["sum", "count"])
hourly.columns = ["fraud_count", "total_count"]
hourly["fraud_rate"] = hourly["fraud_count"] / hourly["total_count"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(hourly.index, hourly["total_count"],
             color="#2196F3", linewidth=0.8, label="All transactions")
axes[0].plot(hourly.index, hourly["fraud_count"] * 50,   # scaled ×50 to be visible
             color="#F44336", linewidth=0.8, label="Fraud (×50 scale)")
axes[0].set_ylabel("Transaction count")
axes[0].set_title("Transaction Volume and Fraud Over Time", fontweight="bold")
axes[0].legend()

axes[1].plot(hourly.index, hourly["fraud_rate"] * 100,
             color="#F44336", linewidth=0.8)
axes[1].set_ylabel("Fraud rate (%)")
axes[1].set_xlabel("Step (hour)")
axes[1].set_title("Fraud Rate Over Time", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
print(f"Unique origin accounts  (nameOrig) : {df['nameOrig'].nunique():>10,}")
print(f"Unique destination accounts (nameDest): {df['nameDest'].nunique():>8,}")

sample_orig = df["nameOrig"].head(5).tolist()
sample_dest = df["nameDest"].head(5).tolist()
print(f"\nSample nameOrig: {sample_orig}")
print(f"Sample nameDest: {sample_dest}")
print("\n→ Note: 'M' prefix = merchant account (no balance recorded)")
print("→ Both columns will be DROPPED in Stage 3.")


In [ ]:
print("=" * 60)
print("  STAGE 2 COMPLETE — KEY FINDINGS")
print("=" * 60)
print(f"  Total rows        : {n_rows:,}")
print(f"  Total frauds      : {df['isFraud'].sum():,}  ({df['isFraud'].mean()*100:.4f}%)")
print(f"  Missing values    : None")
print(f"  Duplicate rows    : {df.duplicated().sum()}")
print(f"  Rule recall       : {rule_recall:.2f}%  ← ML justification")
print(f"  Fraud in types    : TRANSFER + CASH_OUT only")
print(f"  Memory footprint  : {mem_mb:.1f} MB")
print("=" * 60)
print("\n  NEXT: Stage 3 — Data Cleaning")
print("  (filter to TRANSFER+CASH_OUT, drop leakage columns)")

In [ ]:
os.makedirs("data", exist_ok=True)

PARQUET_RAW = "data/paysim_raw.parquet"
df.to_parquet(PARQUET_RAW, index=False)

size_mb = os.path.getsize(PARQUET_RAW) / 1024**2
print(f" Saved df → {PARQUET_RAW}")
print(f"   Rows    : {len(df):,}")
print(f"   File    : {size_mb:.1f} MB  (vs ~470 MB for the original CSV)")
print(f"\n   Open fraud_detection_stage3_4.ipynb to continue.")